# Phase 2 — ViT-Tiny 재현

노트북 11 의 설계를 그대로 옮기되 `1_wsd_grid.ipynb` 와 **같은 교정**을 적용했다
(Stage 2 시드 고정 · probe 분리 · 대칭 sharpness).

**Phase 1 이 Gate B/C 를 통과한 뒤에 돌린다.**
ResNet 결과가 애매하면 "두 아키텍처에서 재현"이라는 논지 자체를 다시 써야 하므로,
크로스-아키텍처 확장 전에 메인 축을 확정하는 게 안전하다.

| | |
|---|---|
| Stage P (사전학습) | 36분 — **한 번만.** 모든 시드가 이 backbone 을 공유 |
| 시드당 | Stage 1 12분 + arm 당 6~12분 |
| 3시드 | 약 2.1시간 |

⚠️ ViT 는 `label_smoothing=0.1` + 64×64 + 파라미터 수가 달라
**sharpness 절대값을 ResNet 과 같은 축에 얹으면 안 된다.** 방향·상대비로만 쓴다.

> ⚠️ 마지막 셀에서 결과와 **사전학습 체크포인트를 반드시 회수**할 것.
> 다시 만들면 backbone 이 달라져 기존 수치와 비교가 끊긴다.

In [ ]:
# ══ 1. 셋업 ═══════════════════════════════════════════════════════════════
import os, sys, math, json, time, glob, random, tarfile, urllib.request
from datetime import datetime
from collections import defaultdict

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

# ── 결과 저장 위치 ────────────────────────────────────────────────────────
# /content 는 세션이 끊기면 통째로 사라진다. 5시간짜리 그리드를 돌린다면
# Drive 에 저장해야 중단 후 재실행 시 끝난 arm 을 실제로 건너뛸 수 있다.
USE_DRIVE = True          # 짧게 시험만 할 거면 False

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/wsd_sam/out'
    except Exception as e:
        print('Drive 마운트 실패 → /content/out 사용:', e)
        OUT_DIR = '/content/out'
else:
    OUT_DIR = '/content/out'

DATA_DIR = os.environ.get('DATA_DIR', '/content/data')   # 데이터는 매번 받아도 된다
OUT_DIR  = os.environ.get('OUT_DIR', OUT_DIR)
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device,
      f'({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else '')
print('torch  :', torch.__version__)
print('출력   :', OUT_DIR)
if device.type != 'cuda':
    print('\n⚠️ GPU 가 아니다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꿀 것.')

In [ ]:
# ══ 2. 재현성 ═════════════════════════════════════════════════════════════
# 노트북판의 가장 큰 결함이 "Stage 2 에 시드가 없다"는 것이었다.
# sam_off 셀이 RNG 를 소모한 뒤 sam_on 셀이 시작돼, 두 arm 이 서로 다른
# 데이터 순서를 봤다. 아래 세 함수로 그 경로를 원천 차단한다.
def seed_everything(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def loader_generator(seed):
    g = torch.Generator(); g.manual_seed(seed); return g

def worker_init(worker_id):
    random.seed(torch.initial_seed() % 2**32 + worker_id)

print('시드 유틸 준비 완료')

In [ ]:
# ══ 3. 데이터 (Tiny-ImageNet + CIFAR-10) ═════════════════════════════════
from PIL import Image
from torch.utils.data import Dataset
import zipfile

TIN_URL = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
CIFAR_URL = 'https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz'
TIN_MEAN, TIN_STD = (0.4802, 0.4481, 0.3975), (0.2770, 0.2691, 0.2821)
C10_MEAN, C10_STD = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
IMG_SIZE = 64        # ViT 입력. CIFAR-10(32)도 여기 맞춰 리사이즈해야
                     # 사전학습 backbone(patch_embed, pos_embed)을 그대로 재사용한다


class TinyImageNetVal(Dataset):
    """val 은 클래스별로 나뉘어 있지 않고 val_annotations.txt 에 매핑이 있다."""
    def __init__(self, root, class_to_idx, transform=None):
        self.transform = transform; self.samples = []
        with open(os.path.join(root, 'val', 'val_annotations.txt')) as f:
            for line in f:
                q = line.strip().split('\t')
                if len(q) >= 2 and q[1] in class_to_idx:
                    self.samples.append((os.path.join(root,'val','images',q[0]),
                                         class_to_idx[q[1]]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert('RGB')      # 흑백 이미지가 섞여 있다
        return (self.transform(img) if self.transform else img), y


def make_transforms(mean, std, train, strong_aug):
    if not train:
        return transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean, std)])
    ops = [transforms.Resize((IMG_SIZE, IMG_SIZE)),
           transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE//8),
           transforms.RandomHorizontalFlip()]
    if strong_aug:
        # ViT 는 inductive bias 가 없어 강한 증강이 사실상 필수 (DeiT 레시피)
        ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
    ops += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(ops)


def prepare_tin():
    root = os.path.join(DATA_DIR, 'tiny-imagenet-200')
    if not os.path.exists(root):
        z = os.path.join(DATA_DIR, 'tiny-imagenet-200.zip')
        if not os.path.exists(z):
            print('Tiny-ImageNet 다운로드 중... (248MB)', flush=True)
            urllib.request.urlretrieve(TIN_URL, z)
        print('압축 해제 중...', flush=True)
        with zipfile.ZipFile(z) as f: f.extractall(DATA_DIR)
    return root


def prepare_cifar():
    root = os.path.join(DATA_DIR, 'cifar10')
    if not os.path.exists(root):
        t = os.path.join(DATA_DIR, 'cifar10.tgz')
        if not os.path.exists(t):
            print('CIFAR-10 다운로드 중...', flush=True)
            urllib.request.urlretrieve(CIFAR_URL, t)
        with tarfile.open(t, 'r:gz') as tar: tar.extractall(path=DATA_DIR, filter='data')
    return root


def make_probe(dataset, n=384, batch_size=128, probe_seed=1234):
    g = torch.Generator().manual_seed(probe_seed)
    idx = torch.randperm(len(dataset), generator=g)[:n].tolist()
    return [(x.to(device), y.to(device))
            for x, y in DataLoader(Subset(dataset, idx), batch_size=batch_size)]


def cifar_loaders(seed, batch_size=256, num_workers=2):
    root = prepare_cifar()
    tr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(C10_MEAN, C10_STD, True, False))
    te = datasets.ImageFolder(os.path.join(root,'test'),
                              transform=make_transforms(C10_MEAN, C10_STD, False, False))
    pr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(C10_MEAN, C10_STD, False, False))
    tl = DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                    pin_memory=True, drop_last=True,
                    generator=loader_generator(seed), worker_init_fn=worker_init)
    vl = DataLoader(te, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True)
    return tl, vl, pr

print('데이터 함수 준비 완료')

In [ ]:
# ══ 4. ViT-Tiny + Lion + SAM ══════════════════════════════════════════════
class PatchEmbed(nn.Module):
    def __init__(self, img_size=64, patch=8, in_ch=3, dim=192):
        super().__init__()
        self.num_patches = (img_size//patch)**2
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        h = int(dim*mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim,h), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(h,dim), nn.Dropout(drop))
    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.mlp(self.norm2(x))

class ViT(nn.Module):
    """ViT-Tiny 급. 64x64 / patch 8 → 64 토큰 + cls = 65"""
    def __init__(self, num_classes=200, img_size=64, patch=8, dim=192,
                 depth=6, heads=3, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch, 3, dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1,1,dim))
        self.pos_embed = nn.Parameter(torch.zeros(1,n+1,dim))
        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([Block(dim,heads,mlp_ratio,drop) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim); self.head = nn.Linear(dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        B = x.size(0)
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], 1) + self.pos_embed
        x = self.pos_drop(x)
        for b in self.blocks: x = b(x)
        return self.head(self.norm(x)[:, 0])

def build_vit(num_classes): return ViT(num_classes=num_classes).to(device)

def load_backbone(model, sd):
    """head 를 뺀 backbone 만 이식 (200클래스 → 10클래스 전이)"""
    bb = {k: v for k, v in sd.items() if not k.startswith('head.')}
    missing, _ = model.load_state_dict(bb, strict=False)
    print('   backbone 이식 — 새로 초기화:', missing, flush=True)
    return model


class Lion(optim.Optimizer):
    def __init__(self, params, lr=2e-4, betas=(0.9,0.99), weight_decay=0.5):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))
    @torch.no_grad()
    def step(self, closure=None):
        for g_ in self.param_groups:
            lr, wd = g_['lr'], g_['weight_decay']; b1, b2 = g_['betas']
            for p in g_['params']:
                if p.grad is None: continue
                st = self.state[p]
                if 'exp_avg' not in st: st['exp_avg'] = torch.zeros_like(p)
                m, g = st['exp_avg'], p.grad
                if wd: p.mul_(1 - lr*wd)
                p.add_(m.mul(b1).add_(g, alpha=1-b1).sign_(), alpha=-lr)
                m.mul_(b2).add_(g, alpha=1-b2)


class SAMAscent:
    def __init__(self, params, rho=0.0):
        self.params = [p for p in params if p.requires_grad]
        self.rho = rho; self._e_w = {}
    def set_rho(self, r): self.rho = r
    @torch.no_grad()
    def grad_norm(self):
        gs = [p.grad.norm(p=2) for p in self.params if p.grad is not None]
        return torch.norm(torch.stack(gs), p=2) if gs else torch.tensor(0.0)
    @torch.no_grad()
    def ascent(self):
        if self.rho <= 0: return False
        gn = self.grad_norm()
        if gn.item() < 1e-12: return False
        sc = self.rho/(gn+1e-12); self._e_w.clear()
        for p in self.params:
            if p.grad is None: continue
            e = p.grad*sc.to(p.device); p.add_(e); self._e_w[p] = e
        return True
    @torch.no_grad()
    def restore(self):
        for p, e in self._e_w.items(): p.sub_(e)
        self._e_w.clear()

print('ViT / Lion / SAMAscent 준비 완료')

In [ ]:
# ══ 5. 스케줄 + sharpness ═════════════════════════════════════════════════
def wsd_lr(epoch, total, peak, warm, decay):
    se = total - decay
    if epoch < warm: return peak*(epoch+1)/warm
    if epoch < se: return peak
    return peak*(1.0 - min(1.0, max(0.0, (epoch-se+1)/decay)))

@torch.no_grad()
def _perturb(e_ws, sign):
    for p, e in e_ws: p.add_(e, alpha=sign)

def measure_sharpness(model, criterion, probe, rho=0.05):
    """one_sided = L(w+e)−L(w)  ≈ rho*||g|| + 곡률   (1차항이 grad norm)
       symmetric = (L(w+e)+L(w−e))/2 − L(w)          (1차항 상쇄, 곡률만)
       grad_norm = Gate A 검증용 원자료"""
    was = model.training; model.eval()
    one, sym, gns = [], [], []
    for x, y in probe:
        with torch.no_grad(): l0 = criterion(model(x), y).item()
        model.zero_grad(set_to_none=True); criterion(model(x), y).backward()
        gs = [p.grad.norm(p=2) for p in model.parameters() if p.grad is not None]
        gn = torch.norm(torch.stack(gs), p=2); gns.append(gn.item())
        sc = rho/(gn+1e-12)
        e_ws = [(p, p.grad*sc) for p in model.parameters() if p.grad is not None]
        with torch.no_grad():
            _perturb(e_ws, +1.0); lp = criterion(model(x), y).item()
            _perturb(e_ws, -2.0); lm = criterion(model(x), y).item()
            _perturb(e_ws, +1.0)
        model.zero_grad(set_to_none=True)
        one.append(lp-l0); sym.append(0.5*(lp+lm)-l0)
    if was: model.train()
    n = len(one)
    return sum(one)/n, sum(sym)/n, sum(gns)/n

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        t += y.size(0); c += model(x).max(1)[1].eq(y).sum().item()
    return 100.0*c/t

def new_history():
    keys = ['epoch','phase','lr','rho','train_loss','train_acc','test_acc',
            'grad_norm','ascent_gap','time']
    return ({k: [] for k in keys},
            {'epoch': [], 'one_sided': [], 'symmetric': [], 'probe_grad_norm': []})

print('스케줄 / sharpness 준비 완료')

In [ ]:
# ══ 6. 실시간 그래프 ══════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from IPython.display import clear_output

LIVE = {'on': True, 'every': 1}       # every=5 로 두면 5에폭마다 갱신

def live_plot(tag, history, sharp):
    if not LIVE['on'] or len(history['epoch']) % LIVE['every']: return
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history['epoch'], history['train_acc'], lw=1.2, label='train')
    ax[0].plot(history['epoch'], history['test_acc'],  lw=1.6, label='test')
    ax[0].set_xlabel('epoch'); ax[0].set_ylabel('accuracy (%)')
    ax[0].set_title(f'{tag} — test {history["test_acc"][-1]:.2f}%')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    if sharp['epoch']:
        ax[1].plot(sharp['epoch'], sharp['one_sided'], 'o-', ms=3,
                   color='tab:orange', label='one-sided')
        ax[1].plot(sharp['epoch'], sharp['symmetric'], 's-', ms=3,
                   color='tab:purple', label='symmetric (curvature)')
        ax[1].set_xlabel('epoch'); ax[1].set_ylabel('sharpness')
        ax[1].set_title('sharpness (lower = flatter)')
        ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    ax[2].plot(history['epoch'], history['grad_norm'], color='tab:brown', lw=1.2)
    ax[2].set_xlabel('epoch'); ax[2].set_ylabel('grad norm', color='tab:brown')
    ax[2].grid(alpha=.3); ax[2].set_title('grad norm & rho')
    a2 = ax[2].twinx(); a2.plot(history['epoch'], history['rho'],
                                color='tab:green', lw=1.4)
    a2.set_ylabel('rho', color='tab:green')
    plt.tight_layout(); plt.show()
    print(f'[{tag}] epoch {history["epoch"][-1]} | '
          f'loss {history["train_loss"][-1]:.4f} | '
          f'test {history["test_acc"][-1]:.2f}% | '
          f'{sum(history["time"])/60:.1f}분 경과', flush=True)

print('실시간 그래프 준비 완료 (끄려면 LIVE["on"] = False)')

In [ ]:
# ══ 7. 학습 루프 ══════════════════════════════════════════════════════════
def run_epochs(model, opt, sam, crit, tl, vl, probe, epoch_range, cfg,
               hist, sharp, rho_fn, tag):
    se = cfg['total_epochs'] - cfg['decay_epochs']; last = epoch_range[-1]
    for epoch in epoch_range:
        lr_now = wsd_lr(epoch, cfg['total_epochs'], cfg['peak_lr'],
                        cfg['warmup_epochs'], cfg['decay_epochs'])
        for g in opt.param_groups: g['lr'] = lr_now
        rho_now = rho_fn(epoch) if rho_fn else 0.0
        sam.set_rho(rho_now)
        model.train()
        rl = rg = rn = 0.0; c = t = nb = 0; t0 = time.time()
        for x, y in tl:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            o = model(x); loss = crit(o, y); loss.backward()
            t += y.size(0); c += o.max(1)[1].eq(y).sum().item()
            rn += sam.grad_norm().item()
            gap = 0.0
            if sam.ascent():
                opt.zero_grad(set_to_none=True)
                la = crit(model(x), y); la.backward(); gap = la.item()-loss.item()
                sam.restore()
            if cfg['grad_clip']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
            opt.step(); rl += loss.item(); rg += gap; nb += 1
        dt = time.time()-t0
        te = evaluate(model, vl)
        phase = ('warmup' if epoch < cfg['warmup_epochs']
                 else 'stable' if epoch < se else 'decay')
        for k, v in [('epoch',epoch+1),('phase',phase),('lr',lr_now),('rho',rho_now),
                     ('train_loss',rl/nb),('train_acc',100.0*c/t),('test_acc',te),
                     ('grad_norm',rn/nb),('ascent_gap',rg/nb),('time',dt)]:
            hist[k].append(v)
        extra = ''
        if (epoch+1) % cfg['sharpness_interval'] == 0 or epoch == last:
            o_, s_, g_ = measure_sharpness(model, crit, probe, cfg['sharpness_rho'])
            sharp['epoch'].append(epoch+1); sharp['one_sided'].append(o_)
            sharp['symmetric'].append(s_); sharp['probe_grad_norm'].append(g_)
            extra = f' | Sharp1 {o_:.4f} | SharpSym {s_:.6f}'
        print(f'[{tag}|{phase:6s}] Ep [{epoch+1}/{cfg["total_epochs"]}] | {dt:.1f}s | '
              f'Loss {rl/nb:.4f} | Train {100.0*c/t:.2f}% | Eval {te:.2f}% | '
              f'lr {lr_now:.2e} | rho {rho_now:.4f}{extra}', flush=True)
        try: live_plot(tag, hist, sharp)
        except Exception as e: print('  (그래프 오류 무시:', e, ')', flush=True)

print('학습 루프 준비 완료')

In [ ]:
# ══ 8. Stage P / 1 / 2 ════════════════════════════════════════════════════
def stage_pretrain(cfg, pretrain_seed=42, force=False):
    """Tiny-ImageNet 사전학습. 모든 시드가 이 backbone 하나를 공유한다.
    연구 대상은 decay 구간이지 사전학습 분산이 아니므로 시드마다 다시 하지 않는다."""
    path = os.path.join(OUT_DIR, 'vit_pretrain_tinyimagenet.pt')
    if os.path.exists(path) and not force:
        print(f'Stage P 캐시 재사용: {path}', flush=True); return path
    root = prepare_tin()
    tr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(TIN_MEAN, TIN_STD, True, True))
    pr = datasets.ImageFolder(os.path.join(root,'train'),
                              transform=make_transforms(TIN_MEAN, TIN_STD, False, False))
    va = TinyImageNetVal(root, tr.class_to_idx,
                         transform=make_transforms(TIN_MEAN, TIN_STD, False, False))
    seed_everything(pretrain_seed)
    tl = DataLoader(tr, batch_size=cfg['batch_size'], shuffle=True, num_workers=2,
                    pin_memory=True, drop_last=True,
                    generator=loader_generator(pretrain_seed), worker_init_fn=worker_init)
    vl = DataLoader(va, batch_size=cfg['batch_size'], shuffle=False,
                    num_workers=2, pin_memory=True)
    pcfg = dict(total_epochs=cfg['pre_epochs'], warmup_epochs=3, decay_epochs=5,
                peak_lr=cfg['pre_lr'], grad_clip=1.0,
                sharpness_interval=5, sharpness_rho=cfg['sharpness_rho'])
    model = build_vit(len(tr.classes))
    opt = Lion(model.parameters(), lr=cfg['pre_lr'], weight_decay=cfg['weight_decay'])
    sam = SAMAscent(model.parameters())
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)   # ViT 엔 label smoothing 이 효과적
    hist, sharp = new_history()
    print(f'\n=== Stage P — Tiny-ImageNet ({cfg["pre_epochs"]} epochs) ===\n', flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, make_probe(pr),
               range(cfg['pre_epochs']), pcfg, hist, sharp, None, 'P')
    torch.save({'model': model.state_dict(), 'cfg': pcfg,
                'num_classes': len(tr.classes), 'seed': pretrain_seed,
                'history': hist, 'sharpness_history': sharp}, path)
    print(f'\nStage P 저장: {path}', flush=True)
    print(f'  val acc {hist["test_acc"][-1]:.2f}% (200클래스 랜덤 0.5%) | '
          f'{sum(hist["time"])/60:.1f}분', flush=True)
    return path


def stage1(seed, cfg, pre_path, force=False):
    path = os.path.join(OUT_DIR, f'vit_stage1_seed{seed}.pt')
    if os.path.exists(path) and not force:
        print(f'Stage 1 캐시 재사용: {path}', flush=True); return path
    seed_everything(seed)
    tl, vl, pr = cifar_loaders(seed, cfg['batch_size'])
    model = build_vit(10)
    print('사전학습 backbone 이식 중...', flush=True)
    load_backbone(model, torch.load(pre_path, map_location=device,
                                    weights_only=False)['model'])
    opt = Lion(model.parameters(), lr=cfg['ft_lr'], weight_decay=cfg['weight_decay'])
    sam = SAMAscent(model.parameters())
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist, sharp = new_history()
    print(f'\n=== Stage 1 (seed {seed}) — CIFAR-10 {cfg["stable_end"]} epochs ===\n',
          flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, make_probe(pr),
               range(cfg['stable_end']), cfg, hist, sharp, None, f'S1-s{seed}')
    torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(),
                'cfg': cfg, 'seed': seed, 'history': hist,
                'sharpness_history': sharp}, path)
    print(f'\nStage 1 저장: {path}', flush=True)
    return path


def stage2(seed, rho, cfg, ckpt_path):
    tag = f'sam{rho:g}' if rho > 0 else 'samoff'
    arm = f'vit_seed{seed}_{tag}_d{cfg["decay_epochs"]}'
    out = os.path.join(OUT_DIR, f'arm_{arm}.json')
    if os.path.exists(out):
        print(f'skip {arm} (이미 있음)', flush=True)
        return json.load(open(out, encoding='utf-8'))
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    se = ck['cfg']['stable_end']

    # ★ 노트북판에 없던 부분 — 모든 arm 이 같은 데이터 순서를 보게 한다
    seed_everything(seed + 10_000)
    tl, vl, pr = cifar_loaders(seed + 10_000, cfg['batch_size'])

    model = build_vit(10); model.load_state_dict(ck['model'])
    opt = Lion(model.parameters(), lr=cfg['ft_lr'], weight_decay=cfg['weight_decay'])
    opt.load_state_dict(ck['optimizer'])            # 모멘텀까지 복원
    sam = SAMAscent(model.parameters())
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {k: list(v) for k, v in ck['history'].items()}
    sharp = {k: list(v) for k, v in ck['sharpness_history'].items()}

    def rho_fn(ep):
        if rho <= 0: return 0.0
        return rho * min(1.0, max(0.0, (ep-se+1)/max(1, cfg['rho_ramp_epochs'])))

    a0, o0, s0 = hist['test_acc'][-1], sharp['one_sided'][-1], sharp['symmetric'][-1]
    print(f'\n=== Stage 2 [{arm}] — 출발점 ep{se}: Test {a0:.2f}% | '
          f'SharpSym {s0:.6f} ===\n', flush=True)
    run_epochs(model, opt, sam, crit, tl, vl, make_probe(pr),
               range(se, cfg['total_epochs']), cfg, hist, sharp, rho_fn, f'S2-{arm}')

    best = max(hist['test_acc'])
    rec = {'arm': arm, 'seed': seed, 'arch': 'vit_tiny', 'rho_max': rho,
           'decay_epochs': cfg['decay_epochs'], 'stable_end': se, 'cfg': cfg,
           'results': {'decay_start_test_acc': a0, 'decay_start_one_sided': o0,
                       'decay_start_symmetric': s0,
                       'final_train_acc': hist['train_acc'][-1],
                       'final_test_acc': hist['test_acc'][-1], 'best_test_acc': best,
                       'final_gap': hist['train_acc'][-1]-hist['test_acc'][-1],
                       'final_one_sided': sharp['one_sided'][-1],
                       'final_symmetric': sharp['symmetric'][-1],
                       'final_probe_grad_norm': sharp['probe_grad_norm'][-1],
                       'decay_time_min': sum(hist['time'][se:])/60},
           'history': hist, 'sharpness_history': sharp,
           'timestamp': datetime.now().isoformat(timespec='seconds')}
    json.dump(rec, open(out, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    r = rec['results']
    print(f'\n=== [{arm}] 완료 ===', flush=True)
    print(f'  Test     {a0:.2f}% -> {r["final_test_acc"]:.2f}% '
          f'({r["final_test_acc"]-a0:+.2f}%p)', flush=True)
    print(f'  SharpSym {s0:.6f} -> {r["final_symmetric"]:.6f}', flush=True)
    print(f'  {r["decay_time_min"]:.1f}분 | 저장: {out}', flush=True)
    return rec

print('Stage 함수 준비 완료')

In [ ]:
# ══ 9. 설정 ═══════════════════════════════════════════════════════════════
SEEDS = [42, 43, 44]
RHOS  = [0.0, 0.05]

CFG = dict(stable_end=20, decay_epochs=10, warmup_epochs=2, total_epochs=30,
           pre_epochs=30, pre_lr=3e-4, ft_lr=1e-4, weight_decay=0.5,
           grad_clip=1.0, batch_size=256, rho_ramp_epochs=3,
           sharpness_interval=2, sharpness_rho=0.05)

print(f'Stage P 1회 + arm {len(SEEDS)*len(RHOS)}개 | 예상 약 2.1시간')

In [ ]:
# ══ 10. 실행 ══════════════════════════════════════════════════════════════
pre = stage_pretrain(CFG)                 # 36분, 한 번만
for seed in SEEDS:
    ck = stage1(seed, CFG, pre)
    for rho in RHOS:
        stage2(seed, rho, CFG, ck)
print('\n완료')

In [ ]:
# ══ 11. 결과 표 ═══════════════════════════════════════════════════════════
arms = [json.load(open(p, encoding='utf-8'))
        for p in sorted(glob.glob(os.path.join(OUT_DIR, 'arm_vit_*.json')))]
print(f'{"seed":>5}{"rho":>7}{"acc0":>8}{"acc1":>8}{"one":>9}{"sym":>12}{"|g|":>8}')
print('-'*57)
for a in sorted(arms, key=lambda x: (x['seed'], x['rho_max'])):
    r = a['results']
    print(f'{a["seed"]:>5}{a["rho_max"]:>7.2f}{r["decay_start_test_acc"]:>8.2f}'
          f'{r["final_test_acc"]:>8.2f}{r["final_one_sided"]:>9.4f}'
          f'{r["final_symmetric"]:>12.6f}{r["final_probe_grad_norm"]:>8.3f}')
print('-'*57)

# 시드별 paired 비교 (ResNet 의 Gate B 와 같은 형태)
from collections import defaultdict
by = defaultdict(dict)
for a in arms: by[a['seed']][a['rho_max']] = a['results']
pairs = [(s, d[0.0]['final_symmetric'], d[0.05]['final_symmetric'],
          d[0.0]['final_test_acc'], d[0.05]['final_test_acc'])
         for s, d in sorted(by.items()) if 0.0 in d and 0.05 in d]
if pairs:
    k = sum(1 for _,so,sn,_,_ in pairs if sn < so)
    print(f'\nsam_on 의 SharpSym 이 더 낮은 시드: {k}/{len(pairs)}')
    for s,so,sn,ao,an in pairs:
        print(f'  seed {s}: SharpSym {so:.6f} -> {sn:.6f} ({sn-so:+.6f}) | '
              f'acc {ao:.2f} -> {an:.2f} ({an-ao:+.2f}%p)')

In [ ]:
# ══ ★ 회수 (세션 끄기 전 반드시 실행) ═════════════════════════════════════
# Colab VM 은 임시다. Kaggle 에서 이미 한 번 전부 날렸다 (PROGRESS.md §8).
arc = os.path.join(OUT_DIR, 'phase2_vit_results.tar.gz')
with tarfile.open(arc, 'w:gz') as t:
    for f in sorted(glob.glob(os.path.join(OUT_DIR, '*.json'))
                    + glob.glob(os.path.join(OUT_DIR, '*.png'))):
        t.add(f, arcname=os.path.basename(f))
print(f'{arc}  ({os.path.getsize(arc)/1024:.0f} KB)')
try:
    from google.colab import files; files.download(arc)
except Exception as e:
    print('브라우저 다운로드 불가:', e)
    print('CLI 로:  colab download -s <세션>', arc, './results/')

# ★ 사전학습 backbone 은 용량이 커서 위 아카이브에 안 들어간다. 따로 받을 것.
pre_ck = os.path.join(OUT_DIR, 'vit_pretrain_tinyimagenet.pt')
print(f'\n사전학습 체크포인트: {pre_ck} '
      f'({os.path.getsize(pre_ck)/1024**2:.0f} MB)' if os.path.exists(pre_ck) else '')
try:
    from google.colab import files; files.download(pre_ck)
except Exception as e:
    print('CLI 로:  colab download -s <세션>', pre_ck, './results/')